# 실습 5-1 · 데이터 전처리와 피처 설계

입력: `../data/unsw_nb15_sample.csv` (UNSW-NB15 원본 1번 파일의 첫 20분 표본, 값은 원본 그대로)
출력: `../data/processed/` 의 train.csv · test.csv · later.csv (05b · 05c · 05s · 06s가 읽음)

순서: [1] 불러오기 → [2] 품질 점검 → [3] 정제 → [4] 7:3 분할 → [5] rate 계산 → [6] 피처 계산 → [7] 피처 점검 → [8] 저장

정답(attack_cat, label)과 IP · 포트 · 시각은 모델 입력에 넣지 않습니다. [4] 분할 이후의 기준값은 학습셋에서만 구합니다(평가셋 정보 누수 방지).

## [1] 불러오기

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/unsw_nb15_sample.csv", dtype={"dsport": str})
print(df.shape)
df.head()

(5050, 19)


,srcip,sport,dstip,dsport,proto,state,service,dur,sbytes,dbytes,Spkts,Dpkts,Stime,Ltime,ct_srv_dst,ct_dst_ltm,ct_src_dport_ltm,attack_cat,Label
0,192.168.241.243,259,192.168.241.243,49320,icmp,URH,-,0.000000,1780,0,5,0,1421927381,1421927439,2,2,1,NaN,0
1,59.166.0.6,2142,149.171.126.4,53,udp,CON,dns,0.001134,132,164,2,2,1421927414,1421927414,7,1,2,NaN,0
2,59.166.0.5,6645,149.171.126.2,80,tcp,FIN,http,29.268066,1052,824,8,10,1421927414,1421927443,10,7,4,NaN,0
3,59.166.0.6,50089,149.171.126.9,80,tcp,FIN,http,28.680214,1052,824,8,10,1421927414,1421927443,3,5,4,NaN,0
4,59.166.0.1,41214,149.171.126.6,10211,tcp,FIN,-,30.733423,4656,3080,28,30,1421927415,1421927446,11,11,1,NaN,0


| 컬럼 | 뜻 |
|---|---|
| srcip:sport / dstip:dsport | 출발지 · 목적지 IP와 포트 (입력 제외) |
| proto / state | 프로토콜 / 연결 상태 (FIN 정상 종료, INT 응답 없음, CON 연결 중) |
| service | 서비스 (http, dns, smtp … , `-` = 식별 안 됨) |
| dur | 지속시간(초) |
| Spkts / Dpkts, sbytes / dbytes | 송신 / 수신 패킷 수, 바이트 수 |
| ct_srv_dst / ct_dst_ltm / ct_src_dport_ltm | 최근 100개 연결 중 같은 서비스·목적지 / 같은 목적지 / 같은 출발지·목적지 포트 연결 수 |
| attack_cat / Label | 정답: 공격 유형(정상은 빈 칸) / 0 정상 1 공격 (입력 제외) |

## [2] 품질 점검: 원본에 있는 문제를 그대로 확인

In [8]:
print("중복 행             :", df.duplicated().sum())
print("결측                :", df.isna().sum()[df.isna().sum() > 0].to_dict(), "(정상은 attack_cat이 빈 칸)")
print("숫자가 아닌 포트     :", df.loc[~df["dsport"].str.isnumeric(), "dsport"].unique().tolist())
print("dur = 0 인 연결      :", (df["dur"] == 0).sum())
print("service '-' 비율     :", round((df["service"] == "-").mean(), 2))

중복 행             : 21
결측                : {'attack_cat': 4800} (정상은 attack_cat이 빈 칸)
숫자가 아닌 포트     : ['0xc0a8']
dur = 0 인 연결      : 27
service '-' 비율     : 0.55


## [3] 정제: 중복 제거, 빈 정답 = Normal, 이름 통일

In [11]:
df = df.drop_duplicates().copy()
df["attack_cat"] = df["attack_cat"].fillna("Normal").str.strip()
df = df.rename(columns={"Spkts": "spkts", "Dpkts": "dpkts", "Label": "label"})
print(df.head(3))
# df.insert(0, "id", range(1, len(df) + 1))
print(len(df), "행")
df["attack_cat"].value_counts()

   id            srcip  sport            dstip dsport proto state service  \
0   1  192.168.241.243    259  192.168.241.243  49320  icmp   URH       -   
1   2       59.166.0.6   2142    149.171.126.4     53   udp   CON     dns   
2   3       59.166.0.5   6645    149.171.126.2     80   tcp   FIN    http   

         dur  sbytes  dbytes  spkts  dpkts       Stime       Ltime  \
0   0.000000    1780       0      5      0  1421927381  1421927439   
1   0.001134     132     164      2      2  1421927414  1421927414   
2  29.268066    1052     824      8     10  1421927414  1421927443   

   ct_srv_dst  ct_dst_ltm  ct_src_dport_ltm attack_cat  label  
0           2           2                 1     Normal      0  
1           7           1                 2     Normal      0  
2          10           7                 4     Normal      0  
5029 행


attack_cat
Normal            4792
DoS                 89
Exploits            78
Reconnaissance      70
Name: count, dtype: int64

## [4] 학습 · 평가 분할 (7:3, 공격 유형 비율 유지)

In [5]:
train, test = train_test_split(df, test_size=0.3, stratify=df["attack_cat"], random_state=42)   # 분할 시 편향되지 않게 attack_cat 기준으로 데이터의 정답 비율을 훈련 세트와 테스트 세트가 똑같은 비율로 
train, test = train.copy(), test.copy()
print(f"학습 {len(train)}건 / 평가 {len(test)}건")

ValueError: Input contains NaN

## [5] rate = 초당 패킷 수 (dur = 0은 0.001초로 계산)

In [18]:
def add_rate(d):
    d["rate"] = (d["spkts"] + d["dpkts"]) / d["dur"].clip(lower=0.001)
    return d

train, test = add_rate(train), add_rate(test)

## [6] 피처 계산
- 연결 행위: 개수 → 로그 변환
- 규모 · 방향 (서비스 평소 대비): DNS는 작고 메일은 큰 것이 정상이므로 같은 서비스끼리 비교합니다.
  `(값 − 서비스 중앙값) ÷ 서비스 MAD` (MAD = 중앙값에서 떨어진 정도의 중앙값). 중앙값 · MAD는 학습셋에서만 구합니다.

In [19]:
BASE = ["log_sbytes", "log_dbytes", "log_rate", "byte_ratio", "pkt_ratio"]
DESIGNED = ["log_ct_srv_dst", "log_ct_dst_ltm", "log_ct_src_dport_ltm",
            "log_sbytes_vs_svc", "log_dbytes_vs_svc", "log_rate_vs_svc",
            "byte_ratio_vs_svc", "pkt_ratio_vs_svc"]

def base_values(d):
    return pd.DataFrame({
        "log_sbytes": np.log1p(d["sbytes"]), "log_dbytes": np.log1p(d["dbytes"]), "log_rate": np.log1p(d["rate"]),   
                                                                         #log1p는 입력값에 1을 더한 뒤 자연로그를 취하는 연산(log(1 + x))
        "byte_ratio": d["sbytes"] / (d["sbytes"] + d["dbytes"] + 1),     # 송신 바이트 비중
        "pkt_ratio": d["spkts"] / (d["spkts"] + d["dpkts"] + 1),         # 송신 패킷 비중
    }, index=d.index)

b = base_values(train).assign(service=train["service"])
MED = b.groupby("service").median()                                     # 서비스별 중앙값
MAD = b.groupby("service").agg(lambda x: (x - x.median()).abs().median()) + 0.05   #만약 어떤 특정 서비스 그룹에 속한 로그 데이터들이 전부 다 똑같은 값(예: 전부 다 0바이트)을 가졌다면, 그 그룹의 MAD 값(x의 변동성)은 수학적으로 0이 되어버립니다.
                                                                                   #뒤 계산에서 분모가 0이 되지 않도록 +0.05 
def build_features(d):
    f = pd.DataFrame(index=d.index)
    for c in ["ct_srv_dst", "ct_dst_ltm", "ct_src_dport_ltm"]:
        f["log_" + c] = np.log1p(d[c])
    b = base_values(d)
    for c in BASE:             #서비스별 중앙값과 MAD를 활용한 '이상 정도(Robust Z-Score)' 계산
        med = MED[c].reindex(d["service"]).fillna(MED[c].median()).values    
        mad = MAD[c].reindex(d["service"]).fillna(MAD[c].median()).values   
        f[c + "_vs_svc"] = (b[c].values - med) / mad   
                                #현재 연결의 수치(b[c].values)에서 해당 서비스의 평소 중앙값(med)을 뺀 뒤, 변동성(mad)으로 나누어줍니다
    return pd.concat([f, b], axis=1)

ftr, fte = build_features(train), build_features(test)
print("서비스별 평소 송신 바이트 (학습셋 중앙값)")
np.expm1(MED["log_sbytes"]).round(0)    #np.log1p()가 log(1 + x)였다면, np.expm1()은 그 정확히 반대(역함수)인 지수 연산하여 눈으로 검토하기 위함 

서비스별 평소 송신 바이트 (학습셋 중앙값)


service
-            1920.0
dns           146.0
ftp          2934.0
ftp-data      424.0
http         1580.0
irc           928.0
pop3         1631.0
smtp        37372.0
snmp          137.0
ssh          3728.0
ssl          1288.0
Name: log_sbytes, dtype: float64

## [7] 피처 점검: 피처 하나만으로 '그 공격 vs 정상'을 얼마나 나누나 (AUC-ROC)
0.5 = 구분 못함, 1.0 = 완전 구분. 정답은 이 점검에만 씁니다.

In [21]:
allf, alll = pd.concat([ftr, fte]), pd.concat([train["attack_cat"], test["attack_cat"]])
rows = {}
for c in DESIGNED + ["log_sbytes", "log_dbytes", "log_rate"]:
    rows[c] = {}
    for cat in ["DoS", "Reconnaissance", "Exploits"]:
        m = alll.isin(["Normal", cat])
        auc = roc_auc_score(alll[m] == cat, allf.loc[m, c])
        rows[c][cat] = round(max(auc, 1 - auc), 2)    # 값이 작을수록 공격인 경우도 같은 기준으로
pd.DataFrame(rows).T

,DoS,Reconnaissance,Exploits
log_ct_srv_dst,0.83,0.83,0.85
log_ct_dst_ltm,0.85,0.88,0.85
log_ct_src_dport_ltm,0.59,0.60,0.58
log_sbytes_vs_svc,0.65,0.91,0.73
log_dbytes_vs_svc,0.75,0.88,0.83
log_rate_vs_svc,0.67,0.69,0.66
byte_ratio_vs_svc,0.78,0.91,0.71
pkt_ratio_vs_svc,0.81,0.94,0.80
log_sbytes,0.50,0.74,0.53
log_dbytes,0.59,0.82,0.56


## [8] 저장 (+ 05c용 60분 뒤 구간도 같은 기준값으로 변환)

In [22]:
os.makedirs("../data/processed", exist_ok=True)
KEEP = ["id", "srcip", "sport", "dstip", "dsport", "Stime", "proto", "state", "service",
        "dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "ct_srv_dst", "ct_dst_ltm", "ct_src_dport_ltm",
        "attack_cat", "label"]
pd.concat([train[KEEP], ftr], axis=1).to_csv("../data/processed/train.csv", index=False)
pd.concat([test[KEEP], fte], axis=1).to_csv("../data/processed/test.csv", index=False)

later = pd.read_csv("../data/unsw_nb15_sample_later.csv", dtype={"dsport": str}).drop_duplicates()
later = later.rename(columns={"Spkts": "spkts", "Dpkts": "dpkts", "Label": "label"})
later["attack_cat"] = later["attack_cat"].fillna("Normal").str.strip()
later = add_rate(later)
pd.concat([later, build_features(later)], axis=1).to_csv("../data/processed/later.csv", index=False)
print("저장: data/processed/train.csv, test.csv, later.csv")

저장: data/processed/train.csv, test.csv, later.csv
